# 02 Video Preprocessing & Dataset

Implementation of the `VideoFrameDataset` for frame-based deepfake detection. Key features:
- **Uniform Sampling**: Extracting a fixed number of frames across the video duration.
- **Padding**: Handling videos shorter than the required frame count.
- **Normalization**: ImageNet mean and standard deviation.

In [ ]:
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

# Config
IMG_SIZE = 224
NUM_FRAMES = 16
BATCH_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. VideoFrameDataset Class

In [ ]:
class VideoFrameDataset(Dataset):
    def __init__(self, dataframe, num_frames=16, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.num_frames = num_frames
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames <= 0:
            cap.release()
            return None
        
        # Uniform sampling indices
        indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                # If frame read fails, append a black frame or the last successful frame
                if len(frames) > 0:
                    frames.append(frames[-1])
                else:
                    frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
                continue
            
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            
            if self.transform:
                frame = self.transform(frame)
            
            frames.append(frame)
            
        cap.release()
        
        # Stack frames into a single tensor: (T, C, H, W)
        return torch.stack(frames)
    
    def __getitem__(self, index):
        row = self.df.iloc[index]
        video_path = row["path"]
        label = torch.tensor(row["label_id"], dtype=torch.float32)
        
        frames = self._get_frames(video_path)
        
        if frames is None:
            # Handle corrupted video by returning a zero tensor
            print(f"Warning: Failed to load video {video_path}")
            frames = torch.zeros((self.num_frames, 3, IMG_SIZE, IMG_SIZE))
            
        return frames, label

## 2. Transforms & DataLoader Test

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Mock dataframe for testing (replace with actual df from exploration if needed)
data_root = Path("../../data/splits/videos")
test_paths = list(data_root.glob("**/*.mp4"))[:5]
test_df = pd.DataFrame({
    "path": [str(p) for p in test_paths],
    "label_id": [0] * len(test_paths)
})

if not test_df.empty:
    dataset = VideoFrameDataset(test_df, num_frames=NUM_FRAMES, transform=transform)
    loader = DataLoader(dataset, batch_size=2, shuffle=True)
    
    frames, labels = next(iter(loader))
    print(f"Batch frames shape: {frames.shape}") # Expected: (Batch, T, C, H, W)
    print(f"Batch labels shape: {labels.shape}")
else:
    print("No videos found for testing logic.")

## 3. Visualize Sample Sampled Frames

In [ ]:
def show_sampled_frames(frames_tensor, num_to_show=5):
    # frames_tensor shape: (T, C, H, W)
    indices = np.linspace(0, len(frames_tensor)-1, num_to_show).astype(int)
    
    plt.figure(figsize=(15, 5))
    for i, idx in enumerate(indices):
        img = frames_tensor[idx].permute(1, 2, 0).numpy()
        # Unnormalize for visualization
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        
        plt.subplot(1, num_to_show, i+1)
        plt.imshow(img)
        plt.title(f"Frame {idx}")
        plt.axis("off")
    plt.show()

if not test_df.empty:
    show_sampled_frames(frames[0])